In [1]:
import os
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from wordcloud import WordCloud
from collections import Counter

### Аренда Циан 2026 год
- 20568 предложений
- адрес
- район
- координаты
- цена аренды в месяц
- количество комнат
- общая площадь, жилая площадь, площадь кухни
- этаж
- число этажей в доме
- год постройки дома
- материал, из которого построен дом

### Продажа Циан 2026 год
- 56590 предложений
- адрес
- координаты
- название ближайшего метро
- цена продажи
- количество комнат
- общая площадь, жилая площадь, площадь кухни
- этаж
- число этажей в доме
- год постройки дома
- материал, из которого построен дом


### Доходы Домклда конец 2025 года
- 8829 домов с данными
- адрес
- координаты
- средний дохож жителей дома
- средняя плата за ЖКУ среди жителей дома
- доля женщин и доля мужчин в доме
- доля жителей с детьми, животными, автомобилем
- возрастная структура: доля людей до 24 лет, от 25 до 34 лет, от 35 до 44 лет, от 45 до 54 лет, от 55 до 64 лет, старше 65 лет

In [3]:
cian_rent = pd.read_csv('Циан//cian_moscow_rent_11-01_2026.csv')
print(f'Циан: {cian_rent.shape}')
cian_buy = pd.read_csv('Циан//cian_moscow_flats_23-03-2026.csv')
print(f'Циан: {cian_buy.shape}')
domclick = pd.read_csv('Домклик//moscow_domclick_new_stats_22_12_2025.csv')
print(f'Домклик: {domclick.shape}')

Циан: (20568, 22)
Циан: (56590, 16)
Домклик: (8829, 17)


### Циан аренда
Считаем среднюю стоимость аренды квадратного метра для каждго объявления, оставляем ее, координаты и количество этажей. Одинаковые координаты будем считать за один и тот же дом. Посчитаем для каждого домма среднюю стоимоть аренды квадратного метра в месяц

In [5]:
cian_rent.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20568 entries, 0 to 20567
Data columns (total 22 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   zone                  20568 non-null  object 
 1   rooms_bucket          20568 non-null  object 
 2   price_bucket          20568 non-null  object 
 3   offer_id              20568 non-null  int64  
 4   how_many_rooms        18213 non-null  float64
 5   price_per_month       20568 non-null  int64  
 6   address               20568 non-null  object 
 7   district              0 non-null      float64
 8   street                20524 non-null  object 
 9   floor                 20568 non-null  int64  
 10  all_floors            20568 non-null  int64  
 11  square_meters         20568 non-null  float64
 12  comm_meters           15388 non-null  float64
 13  kitchen_meters        16830 non-null  float64
 14  commissions           20568 non-null  int64  
 15  year_of_constructio

In [15]:
cian_rent['price_per_m2'] = cian_rent['price_per_month'] / cian_rent['square_meters']
cian_rent = cian_rent[['price_per_m2','all_floors',  'lon', 'lat']].rename(columns={'all_floors':'levels'})

In [16]:
cian_rent

,price_per_m2,levels,lon,lat
0,1555.555556,14,37.169771,55.588389
1,2361.809045,22,37.862705,55.726183
2,1875.000000,24,37.610062,55.626858
3,2300.000000,16,37.709596,55.867041
4,1521.695652,14,37.524013,55.475613
...,...,...,...,...
20563,4285.714286,6,37.591386,55.740421
20564,3125.000000,6,37.627993,55.741713
20565,2642.276423,8,37.640372,55.733863
20566,6000.000000,4,37.591386,55.740421


In [18]:
def get_most_frequent(x):
    """Возвращает самое частое не-None/не-NaN значение из серии"""
    x_clean = x.dropna()
    if len(x_clean) == 0:
        return np.nan
    mode_val = x_clean.mode()
    return mode_val.iloc[0] if not mode_val.empty else np.nan

# Определяем колонки группировки
group_cols = ['lat', 'lon']

# Создаем словарь агрегации
agg_dict = {
    'price_per_m2': 'mean',        # усредняем цену (или 'price' если другое название)
    'levels': get_most_frequent
}

# Для остальных колонок (если они нужны) - берем первое значение или среднее
other_cols = [c for c in cian_rent.columns 
              if c not in group_cols + list(agg_dict.keys())]

for col in other_cols:
    # Для числовых - среднее, для категориальных - первое встретившееся
    if pd.api.types.is_numeric_dtype(cian_rent[col]):
        agg_dict[col] = 'mean'
    else:
        agg_dict[col] = 'first'

# Применяем группировку
cian_rent_final = cian_rent.groupby(
    group_cols, 
    as_index=False
).agg(agg_dict)

print(f"Размер до: {len(cian_rent)}")
print(f"Размер после: {len(cian_rent_final)}")
cian_rent_builds= cian_rent_final[['lat', 'lon', 'price_per_m2', 'levels']]

house_agg = cian_rent.groupby(by=['lat', 'lon'])[['lat']].count().rename(columns={'lat': 'ads_count'}).reset_index()
cian_rent_builds= pd.merge(cian_rent_builds, house_agg, on =['lat', 'lon'], how='left')

cian_rent_builds.shape

Размер до: 20568
Размер после: 9642


(9642, 5)

In [19]:
cian_rent_builds.to_csv('real_estate/cian_rent_builds_11-01_2026.csv', index = False)

### Продажа Циан 2026 год
- 56590 предложений
- адрес
- координаты
- название ближайшего метро
- цена продажи
- количество комнат
- общая площадь, жилая площадь, площадь кухни
- этаж
- число этажей в доме
- год постройки дома
- материал, из которого построен дом

### Продажа Циан 2026 год

In [20]:
cian_buy.columns

Index(['id', 'url', 'roomsCount', 'totalArea', 'livingArea', 'kitchenArea',
       'floorNumber', 'floorsCount', 'buildYear', 'materialType', 'price',
       'latitude', 'longitude', 'metro_name', 'address', 'isNew'],
      dtype='object')

- тип здания (1 -  панельное здание, 2 - монолитное, 3 - кирпичное, 4 - блочное, 5 -деревянное, 0 - другое)
- тип объекта (1 - вторичка, 0 - новостройка)

In [31]:
def to_def_bui_type(value):
    if value == 'panel':
        return 1
    elif value =='monolith':
        return 2
    elif value == 'brick':
        return 3
    elif value == 'block':
        return 4
    elif value == 'wood':
        return 5
    else:
        return 0
cian_buy['building_type'] = cian_buy['materialType'].apply(to_def_bui_type)
cian_buy['object_type'] = cian_buy['isNew'].apply(lambda x: 1 if True else 0)
cian_buy['price_per_m2'] = cian_buy['price']/cian_buy['totalArea']
cian_buy = cian_buy.rename(columns={'floorsCount':'levels','latitude': 'lat', 'longitude': 'lon'})

In [35]:
cian_buy = cian_buy[['lat', 'lon', 'price_per_m2', 'building_type',
       'object_type', 'levels']]
cian_buy.head()

,lat,lon,price_per_m2,building_type,object_type,levels
0,55.536563,37.528370,33802.816901,2,1,9
1,55.632382,37.762542,44502.617801,1,1,17
2,55.877738,37.662677,46195.652174,0,1,14
3,55.745610,37.791504,62068.965517,3,1,5
4,55.808245,37.816603,47493.403694,1,1,17


In [34]:
cian_buy.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 56590 entries, 0 to 56589
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   lat            56590 non-null  float64
 1   lon            56590 non-null  float64
 2   price_per_m2   56590 non-null  float64
 3   building_type  56590 non-null  int64  
 4   object_type    56590 non-null  int64  
 5   levels         56590 non-null  int64  
dtypes: float64(3), int64(3)
memory usage: 2.6 MB


In [37]:
def get_most_frequent(x):
    """Возвращает самое частое не-None/не-NaN значение из серии"""
    x_clean = x.dropna()
    if len(x_clean) == 0:
        return np.nan
    mode_val = x_clean.mode()
    return mode_val.iloc[0] if not mode_val.empty else np.nan

# Определяем колонки группировки
group_cols = ['lat', 'lon']

# Создаем словарь агрегации
agg_dict = {
    'price_per_m2': 'mean',        # усредняем цену (или 'price' если другое название)
    'building_type': get_most_frequent,
    'object_type': get_most_frequent,
    'levels': get_most_frequent
}

# Для остальных колонок (если они нужны) - берем первое значение или среднее
other_cols = [c for c in cian_buy.columns 
              if c not in group_cols + list(agg_dict.keys())]

for col in other_cols:
    # Для числовых - среднее, для категориальных - первое встретившееся
    if pd.api.types.is_numeric_dtype(cian_buy[col]):
        agg_dict[col] = 'mean'
    else:
        agg_dict[col] = 'first'

# Применяем группировку
cian_buy_final = cian_buy.groupby(
    group_cols, 
    as_index=False
).agg(agg_dict)

print(f"Размер до: {len(cian_buy)}")
print(f"Размер после: {len(cian_buy_final)}")
cian_buy_builds = cian_buy_final[['lat', 'lon', 'price_per_m2', 'building_type',
       'object_type', 'levels']]

house_agg = cian_buy.groupby(by=['lat', 'lon'])[['lat']].count().rename(columns={'lat': 'ads_count'}).reset_index()
cian_buy_builds = pd.merge(cian_buy_builds, house_agg, on =['lat', 'lon'], how='left')
cian_buy_builds.shape

Размер до: 56590
Размер после: 14017


(14017, 7)

In [39]:
cian_buy_builds

,lat,lon,price_per_m2,building_type,object_type,levels,ads_count
0,55.211456,37.071235,151093.643199,0,1,5,2
1,55.212283,37.072160,113981.762918,1,1,5,1
2,55.213315,37.076095,107657.232704,0,1,2,1
3,55.214039,37.071513,110294.117647,3,1,5,1
4,55.214270,37.067543,131608.432847,3,1,4,2
...,...,...,...,...,...,...,...
14012,56.007368,37.206234,301983.243567,0,1,18,2
14013,56.008656,37.795951,233775.279637,0,1,14,5
14014,56.008696,37.203692,211200.000000,4,1,12,1
14015,56.009391,37.206207,238095.238095,0,1,12,1


In [41]:
cian_buy_builds.to_csv('real_estate/cian_buy_builds_23-03_2026.csv', index = False)

### Доходы Домклиа конец 2025 года
- 8829 домов с данными
- адрес
- координаты
- средний дохож жителей дома
- средняя плата за ЖКУ среди жителей дома
- доля женщин и доля мужчин в доме
- доля жителей с детьми, животными, автомобилем
- возрастная структура: доля людей до 24 лет, от 25 до 34 лет, от 35 до 44 лет, от 45 до 54 лет, от 55 до 64 лет, старше 65 лет

In [46]:
domclick=pd.read_excel('Домклик/mos_builds_incomes_pop_stats.xlsx')

In [49]:
domclick[['lat','lon', 'address']].value_counts()

lat        lon        address                                                          
55.212663  37.072589  Россия, Москва, пос. Рогово, Школьная улица, 17                      1
55.213344  37.076115  Россия, Москва, пос. Рогово, Школьная улица, 7                       1
55.214299  37.067576  Россия, Москва, пос. Рогово, улица Берёзки, 10 к1                    1
55.214533  37.068111  Россия, Москва, пос. Рогово, улица Берёзки, 8 к1                     1
55.214540  37.072066  Россия, Москва, пос. Рогово, Юбилейная улица, 12 к1                  1
                                                                                          ..
56.006647  37.201108  Россия, Москва, Зеленоград, 2-й микрорайон м-н, Зеленоград, к200а    1
56.007441  37.209246  Россия, Москва, Зеленоград, 1-й микрорайон м-н, Зеленоград, к107Б    1
56.008036  37.208985  Россия, Москва, Зеленоград, 1-й микрорайон м-н, Зеленоград, к106     1
56.009156  37.207276  Россия, Москва, Зеленоград, 1-й микрорайон м-н, Зелен

In [51]:
domclick[['address', 'lat', 'lon',  'year', 
       'total_apartments', 
      'inhab_count', 'under_24_num',
       'between_25_34_num', 'between_35_44_num', 'between_45_54_num',
       'between_55_64_num', 'over_65_num', 'men_num', 'women_num',
       'with_children_num', 'with_pets_num', 'with_car_num',
       'median_income_pop', 'avg_hcs_cost_pop']].to_csv('real_estate/domclick_incomes.csv', index=False)

In [48]:
domclick.columns

Index(['url', 'address', 'lat', 'lon', 'district', 'year', 'wall_type',
       'total_apartments', 'floors', 'median_income', 'avg_hcs_cost',
       'under_24', 'between_25_34', 'between_35_44', 'between_45_54',
       'between_55_64', 'over_65', 'men', 'women', 'with_children',
       'with_pets', 'with_car', 'geometry', 'inhab_count', 'under_24_num',
       'between_25_34_num', 'between_35_44_num', 'between_45_54_num',
       'between_55_64_num', 'over_65_num', 'men_num', 'women_num',
       'with_children_num', 'with_pets_num', 'with_car_num',
       'median_income_pop', 'avg_hcs_cost_pop'],
      dtype='object')

In [52]:
domclick.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9088 entries, 0 to 9087
Data columns (total 37 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   url                9088 non-null   object 
 1   address            9088 non-null   object 
 2   lat                9088 non-null   float64
 3   lon                9088 non-null   float64
 4   district           8056 non-null   object 
 5   year               8909 non-null   float64
 6   wall_type          8734 non-null   object 
 7   total_apartments   6870 non-null   float64
 8   floors             8909 non-null   float64
 9   median_income      9088 non-null   int64  
 10  avg_hcs_cost       8762 non-null   float64
 11  under_24           9038 non-null   float64
 12  between_25_34      9086 non-null   float64
 13  between_35_44      9088 non-null   float64
 14  between_45_54      9088 non-null   float64
 15  between_55_64      9078 non-null   float64
 16  over_65            8988 